In [ ]:
# ================================
# OPTUNA MULTI-OBJECTIVE TUTORIAL
# ================================

# This notebook demonstrates how to optimize a neural network using:
# - Multiple objectives: performance, model size, and FLOPs

import optuna  # Hyperparameter optimization library
import numpy as np  # Numerical operations
import pandas as pd  # Data handling

from sklearn.datasets import load_wine  # Dataset
from sklearn.model_selection import train_test_split  # Data split
from sklearn.pipeline import make_pipeline  # Pipeline
from sklearn.preprocessing import StandardScaler  # Scaling
from sklearn.neural_network import MLPClassifier  # Neural network
from sklearn.metrics import f1_score  # Metric

# ================================
# LOAD DATA
# ================================

# Load dataset
X, y = load_wine(return_X_y=True)

# Split dataset with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ================================
# HELPER FUNCTIONS
# ================================

def count_parameters(model):
    # Count total trainable parameters
    total = 0
    for coef in model.coefs_:
        total += coef.size
    for intercept in model.intercepts_:
        total += intercept.size
    return total

def estimate_flops(model):
    # Approximate FLOPs (forward pass)
    flops = 0
    for coef in model.coefs_:
        flops += 2 * coef.size
    return flops

# ================================
# OBJECTIVE FUNCTION
# ================================

def objective(trial):

    # Suggest architecture
    hidden_layers = trial.suggest_categorical(
        "hidden_layer_sizes",
        [(50,), (100,), (50,50)]
    )

    # Suggest activation
    activation = trial.suggest_categorical(
        "activation", ["relu", "tanh"]
    )

    # Suggest regularization
    alpha = trial.suggest_float(
        "alpha", 1e-5, 1e-1, log=True
    )

    # Build pipeline
    model = make_pipeline(
        StandardScaler(),
        MLPClassifier(
            hidden_layer_sizes=hidden_layers,
            activation=activation,
            alpha=alpha,
            max_iter=500,
            early_stopping=True,
            random_state=42
        )
    )

    # Train model
    model.fit(X_train, y_train)

    # Predict
    preds = model.predict(X_test)

    # Metric
    f1 = f1_score(y_test, preds, average="macro")

    # Extract MLP
    mlp = model.named_steps["mlpclassifier"]

    # Compute objectives
    n_params = count_parameters(mlp)
    flops = estimate_flops(mlp)

    # Return multi-objective values
    return f1, n_params, flops

# ================================
# RUN OPTUNA
# ================================

study = optuna.create_study(
    directions=["maximize", "minimize", "minimize"]
)

study.optimize(objective, n_trials=20)

# ================================
# RESULTS
# ================================

trials = study.best_trials

results = []
for t in trials:
    results.append({
        "F1 Score": t.values[0],
        "Parameters": t.values[1],
        "FLOPs": t.values[2],
        "Architecture": t.params["hidden_layer_sizes"],
        "Activation": t.params["activation"],
        "Alpha": t.params["alpha"]
    })

df = pd.DataFrame(results)

print("\nPareto-optimal solutions:")
print(df.sort_values("F1 Score", ascending=False))

# ================================
# TASK
# ================================

# Modify and experiment:
# 1. Increase number of trials
# 2. Add deeper architectures
# 3. Compare trade-offs between accuracy, size, and FLOPs
# 4. Identify best model under constraints (fastest, smallest, most accurate)
